Due to significant slowdowns with the Soundnet API, we prioritized films with a vote count greater than or equal to 500\. In this notebook, we perform analysis on well\-known composers \(e\.g\., John Williams, Hans Zimmer\) to ensure that this filter isn't too restrictive\. We also inspect a sample of excluded films as a sanity check\. Finally, we create a column to flag records greater than or equal to 500 votes for reference in downstream notebooks\.

In [1]:
# Third-party imports
import pandas as pd

In [2]:
filepath = '/work/pipeline/3.5.Wide_exploded_genre.csv'
df = pd.read_csv(filepath)
df.head()

/tmp/ipykernel_734/3974324424.py:2: DtypeWarning: Columns (45,49,58,84) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)


,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,film_is_history,film_is_horror,film_is_music,film_is_mystery,film_is_romance,film_is_science_fiction,film_is_tv_movie,film_is_thriller,film_is_war,film_is_western
0,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,False,True,False,False,False,False,False,True,False,False
1,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,False,True,False,False,False,False,False,True,False,False
2,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,False,False,False,False,False,False,False,False
3,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,False,False,False,False,False,False,False,False
4,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,False,False,False,False,False,False,False,False


In [3]:
# Narrow down to the columns of interest

columns = [
    'film_vote_count',
    'recording_artist_credit',
    'track_title',
    'album_title',
    'tmdb_id',
    'film_title',
    'film_revenue',
    'film_genres',
    'film_imdb_id',
    'release_group_mbid',
    'release_mbid',
    'rg_primary_type',
    'rg_secondary_types',
    'album_soundtrack_type',
    'is_canonical_soundtrack',
    'is_canonical_songtrack',
    'album_artist_mbids_text',
    'album_artist_names_text',
    'album_artist_types_text',
    'film_soundtrack_composer_raw',
    'composer_names_text',
    'track_number',
    'track_length_ms',
    'recording_mbid',
    'isrcs_text',
    ]

df_view = df[columns]
df_view.head()

,film_vote_count,recording_artist_credit,track_title,album_title,tmdb_id,film_title,film_revenue,film_genres,film_imdb_id,release_group_mbid,...,is_canonical_songtrack,album_artist_mbids_text,album_artist_names_text,album_artist_types_text,film_soundtrack_composer_raw,composer_names_text,track_number,track_length_ms,recording_mbid,isrcs_text
0,13,Philippe Vachey,Alone in the Dark Theme,Alone in the Dark,1027160,Alone in the Dark,0,"Horror, Thriller",tt21859110,fb75d93e-d99d-33c7-bd60-80e2228651d3,...,False,31056e50-d37d-461d-9c56-c1e2ad5ecf58,Philippe Vachey,Person,Christopher French,NaN,2,223066.0,bcb316f0-3f28-4a94-a087-db78d3a51fde,NaN
1,13,Philippe Vachey,[data track],Alone in the Dark,1027160,Alone in the Dark,0,"Horror, Thriller",tt21859110,fb75d93e-d99d-33c7-bd60-80e2228651d3,...,False,31056e50-d37d-461d-9c56-c1e2ad5ecf58,Philippe Vachey,Person,Christopher French,NaN,1,3620786.0,af02c70c-6c69-4c62-9f34-68a898c98402,NaN
2,14,Buc Fifty,Gloc,Shakedown,322855,Shakedown,0,Documentary,tt1794951,65314ddb-ab05-314e-bef9-b1372d825266,...,False,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,Other,Tim Dewit,NaN,12,186106.0,fe0a2712-2fee-4f41-9901-1cff2cf91e77,NaN
3,14,Rob the Viking,Move It Up,Shakedown,322855,Shakedown,0,Documentary,tt1794951,65314ddb-ab05-314e-bef9-b1372d825266,...,False,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,Other,Tim Dewit,NaN,1,158480.0,a48f1589-33cd-457f-978c-f8ef9a818bab,NaN
4,14,Son Doobie,Party People,Shakedown,322855,Shakedown,0,Documentary,tt1794951,65314ddb-ab05-314e-bef9-b1372d825266,...,False,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,Other,Tim Dewit,NaN,2,202053.0,94b1c5ef-adeb-467f-83c3-06c4fa8cfb93,NaN


In [4]:
# View total records and nulls

df_view.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78992 entries, 0 to 78991
Data columns (total 25 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   film_vote_count               78992 non-null  int64  
 1   recording_artist_credit       78992 non-null  object 
 2   track_title                   78992 non-null  object 
 3   album_title                   78992 non-null  object 
 4   tmdb_id                       78992 non-null  int64  
 5   film_title                    78992 non-null  object 
 6   film_revenue                  78992 non-null  int64  
 7   film_genres                   78987 non-null  object 
 8   film_imdb_id                  78610 non-null  object 
 9   release_group_mbid            78992 non-null  object 
 10  release_mbid                  78992 non-null  object 
 11  rg_primary_type               78738 non-null  object 
 12  rg_secondary_types            78992 non-null  object 
 13  a

In [5]:
# Check number of unique films, albums, tracks and artists

unique_film_titles = df_view['film_title'].nunique()
unique_tmdb_id = df_view['tmdb_id'].nunique()
unique_albums = df_view['album_title'].nunique()
unique_tracks = df_view['track_title'].nunique()
unique_artists = df_view['recording_artist_credit'].nunique()

print(f"Number of unique film titles: {unique_film_titles}")
print(f"Number of unique tmdb id's: {unique_tmdb_id}")
print(f"Number of unique album names: {unique_albums}")
print(f"Number of unique track names: {unique_tracks}")
print(f"Number of unique artists: {unique_artists}")

Number of unique film titles: 4368
Number of unique tmdb id's: 4439
Number of unique album names: 4667
Number of unique track names: 69329
Number of unique artists: 11734


In [6]:
# Determine the number of unique track-artist pairs

grouped_df = df_view.groupby([
    'track_title', 
    'recording_artist_credit'
    ]).count()

grouped_df.shape[0]

78180

In [7]:
# Total tracks where artist contains John Williams

williams_df = df_view[df_view["recording_artist_credit"].str.contains("John Williams", case=False, na=False)]
print(f"Number of tracks by John Williams: {len(williams_df)}")
williams_df.head()

Number of tracks by John Williams: 104


,film_vote_count,recording_artist_credit,track_title,album_title,tmdb_id,film_title,film_revenue,film_genres,film_imdb_id,release_group_mbid,...,is_canonical_songtrack,album_artist_mbids_text,album_artist_names_text,album_artist_types_text,film_soundtrack_composer_raw,composer_names_text,track_number,track_length_ms,recording_mbid,isrcs_text
1671,21225,John Williams & Michael Giacchino,Welcome to Jurassic World,Jurassic World: Original Motion Picture Soundt...,135397,Jurassic World,1671537444,"Action, Adventure, Science Fiction, Thriller",tt0369610,6272163e-4080-4806-85ca-5d3c22214d0c,...,False,150e14bd-915d-4574-b574-f77dd7ea72ad,Michael Giacchino,Person,Michael Giacchino,John Williams | Michael Giacchino,3,128000.0,926d37df-7e6b-4be5-94e5-999b16048653,USQ4E1501628
1681,21225,John Williams & Michael Giacchino,As the Jurassic World Turns,Jurassic World: Original Motion Picture Soundt...,135397,Jurassic World,1671537444,"Action, Adventure, Science Fiction, Thriller",tt0369610,6272163e-4080-4806-85ca-5d3c22214d0c,...,False,150e14bd-915d-4574-b574-f77dd7ea72ad,Michael Giacchino,Person,Michael Giacchino,John Williams | Michael Giacchino,4,330000.0,beaadfbf-ab41-465c-9588-4a160cf6533e,USQ4E1501629
1688,21225,John Williams & Michael Giacchino,The Park Is Closed,Jurassic World: Original Motion Picture Soundt...,135397,Jurassic World,1671537444,"Action, Adventure, Science Fiction, Thriller",tt0369610,6272163e-4080-4806-85ca-5d3c22214d0c,...,False,150e14bd-915d-4574-b574-f77dd7ea72ad,Michael Giacchino,Person,Michael Giacchino,John Williams | Michael Giacchino,19,98000.0,dc95cd59-d183-430f-a32a-7a28ed4be930,USQ4E1501644
2557,20206,John Williams,I Can Fly Anything,Star Wars: The Force Awakens: Original Motion ...,140607,Star Wars: The Force Awakens,2068223624,"Adventure, Action, Science Fiction",tt2488496,405fd3c5-0a45-456a-b853-6f734d3b57aa,...,True,53b106e7-0cc6-42cc-ac95-ed8d30a3a98e,John Williams,Person,John Williams,John Williams,3,193000.0,b8ecb00c-4a9d-4c6a-8e0c-d5a049935186,USWD11575423
2558,20206,John Williams,Rey Meets BB‐8,Star Wars: The Force Awakens: Original Motion ...,140607,Star Wars: The Force Awakens,2068223624,"Adventure, Action, Science Fiction",tt2488496,405fd3c5-0a45-456a-b853-6f734d3b57aa,...,True,53b106e7-0cc6-42cc-ac95-ed8d30a3a98e,John Williams,Person,John Williams,John Williams,4,92813.0,a071bacb-b413-4b24-9d98-74e13c26c73a,USWD11575424


In [8]:
# Unique tracks where artist contains John Williams

williams_df = df_view[
    df_view["recording_artist_credit"].str.contains("John Williams", case=False, na=False)
].drop_duplicates(
    subset=["recording_artist_credit", "track_title"]
)
unique_williams = len(williams_df)

print(f"Number of unique John Williams tracks: {unique_williams}")
williams_df.head()

Number of unique John Williams tracks: 103


,film_vote_count,recording_artist_credit,track_title,album_title,tmdb_id,film_title,film_revenue,film_genres,film_imdb_id,release_group_mbid,...,is_canonical_songtrack,album_artist_mbids_text,album_artist_names_text,album_artist_types_text,film_soundtrack_composer_raw,composer_names_text,track_number,track_length_ms,recording_mbid,isrcs_text
1671,21225,John Williams & Michael Giacchino,Welcome to Jurassic World,Jurassic World: Original Motion Picture Soundt...,135397,Jurassic World,1671537444,"Action, Adventure, Science Fiction, Thriller",tt0369610,6272163e-4080-4806-85ca-5d3c22214d0c,...,False,150e14bd-915d-4574-b574-f77dd7ea72ad,Michael Giacchino,Person,Michael Giacchino,John Williams | Michael Giacchino,3,128000.0,926d37df-7e6b-4be5-94e5-999b16048653,USQ4E1501628
1681,21225,John Williams & Michael Giacchino,As the Jurassic World Turns,Jurassic World: Original Motion Picture Soundt...,135397,Jurassic World,1671537444,"Action, Adventure, Science Fiction, Thriller",tt0369610,6272163e-4080-4806-85ca-5d3c22214d0c,...,False,150e14bd-915d-4574-b574-f77dd7ea72ad,Michael Giacchino,Person,Michael Giacchino,John Williams | Michael Giacchino,4,330000.0,beaadfbf-ab41-465c-9588-4a160cf6533e,USQ4E1501629
1688,21225,John Williams & Michael Giacchino,The Park Is Closed,Jurassic World: Original Motion Picture Soundt...,135397,Jurassic World,1671537444,"Action, Adventure, Science Fiction, Thriller",tt0369610,6272163e-4080-4806-85ca-5d3c22214d0c,...,False,150e14bd-915d-4574-b574-f77dd7ea72ad,Michael Giacchino,Person,Michael Giacchino,John Williams | Michael Giacchino,19,98000.0,dc95cd59-d183-430f-a32a-7a28ed4be930,USQ4E1501644
2557,20206,John Williams,I Can Fly Anything,Star Wars: The Force Awakens: Original Motion ...,140607,Star Wars: The Force Awakens,2068223624,"Adventure, Action, Science Fiction",tt2488496,405fd3c5-0a45-456a-b853-6f734d3b57aa,...,True,53b106e7-0cc6-42cc-ac95-ed8d30a3a98e,John Williams,Person,John Williams,John Williams,3,193000.0,b8ecb00c-4a9d-4c6a-8e0c-d5a049935186,USWD11575423
2558,20206,John Williams,Rey Meets BB‐8,Star Wars: The Force Awakens: Original Motion ...,140607,Star Wars: The Force Awakens,2068223624,"Adventure, Action, Science Fiction",tt2488496,405fd3c5-0a45-456a-b853-6f734d3b57aa,...,True,53b106e7-0cc6-42cc-ac95-ed8d30a3a98e,John Williams,Person,John Williams,John Williams,4,92813.0,a071bacb-b413-4b24-9d98-74e13c26c73a,USWD11575424


In [9]:
# Total tracks where artist contains Hans Zimmer

zimmer_df = df_view[df_view["recording_artist_credit"].str.contains("Hans Zimmer", case=False, na=False)]
print(f"Number of tracks by Hans Zimmer: {len(zimmer_df)}")
zimmer_df.head()

Number of tracks by Hans Zimmer: 603


,film_vote_count,recording_artist_credit,track_title,album_title,tmdb_id,film_title,film_revenue,film_genres,film_imdb_id,release_group_mbid,...,is_canonical_songtrack,album_artist_mbids_text,album_artist_names_text,album_artist_types_text,film_soundtrack_composer_raw,composer_names_text,track_number,track_length_ms,recording_mbid,isrcs_text
1059,8290,"Hans Zimmer, Ryan Amon & Steve Mazzaro",The Black Sheep,Chappie,198184,Chappie,104399548,"Crime, Action, Science Fiction",tt1823672,2f4348cd-c7fe-436a-9fa3-1dee76743afb,...,False,e6de1f3b-6484-491c-88dd-6d619f142abc,Hans Zimmer,Person,Hans Zimmer,Hans Zimmer | Ryan Amon | Steve Mazzaro,7,268000.0,400c4f42-8487-4506-8c84-d65d2cef7e94,NaN
1060,8290,Hans Zimmer & Andrew Kawczynski,Welcome to the Real World,Chappie,198184,Chappie,104399548,"Crime, Action, Science Fiction",tt1823672,2f4348cd-c7fe-436a-9fa3-1dee76743afb,...,False,e6de1f3b-6484-491c-88dd-6d619f142abc,Hans Zimmer,Person,Hans Zimmer,Andrew Kawczynski | Hans Zimmer,6,232000.0,666a3c85-0931-4451-924a-3c25ff599e95,NaN
1061,8290,Hans Zimmer & Steve Mazzaro,It's a Dangerous City,Chappie,198184,Chappie,104399548,"Crime, Action, Science Fiction",tt1823672,2f4348cd-c7fe-436a-9fa3-1dee76743afb,...,False,e6de1f3b-6484-491c-88dd-6d619f142abc,Hans Zimmer,Person,Hans Zimmer,Hans Zimmer | Steve Mazzaro,1,129000.0,ad6159b6-e5e4-48ca-9356-86653939e10f,NaN
1062,8290,Hans Zimmer & Steve Mazzaro,You Lied to Me,Chappie,198184,Chappie,104399548,"Crime, Action, Science Fiction",tt1823672,2f4348cd-c7fe-436a-9fa3-1dee76743afb,...,False,e6de1f3b-6484-491c-88dd-6d619f142abc,Hans Zimmer,Person,Hans Zimmer,Hans Zimmer | Steve Mazzaro,11,246000.0,20bea297-6196-4919-8e90-06b00d0b3600,NaN
1063,8290,"Hans Zimmer, Steve Mazzaro & Andrew Kawczynski",Use Your Mind,Chappie,198184,Chappie,104399548,"Crime, Action, Science Fiction",tt1823672,2f4348cd-c7fe-436a-9fa3-1dee76743afb,...,False,e6de1f3b-6484-491c-88dd-6d619f142abc,Hans Zimmer,Person,Hans Zimmer,Andrew Kawczynski | Hans Zimmer | Steve Mazzaro,3,244000.0,5063ec8f-0ca9-4366-abeb-954a6ccaf123,NaN


In [10]:
# Unique tracks where artist contains Hans Zimmer

zimmer_df = df_view[
    df_view["recording_artist_credit"].str.contains("Hans Zimmer", case=False, na=False)
].drop_duplicates(
    subset=["recording_artist_credit", "track_title"]
)

unique_zimmer = len(zimmer_df)
print(f"Number of unique Hans Zimmer tracks: {unique_zimmer}")
zimmer_df.head()

Number of unique Hans Zimmer tracks: 574


,film_vote_count,recording_artist_credit,track_title,album_title,tmdb_id,film_title,film_revenue,film_genres,film_imdb_id,release_group_mbid,...,is_canonical_songtrack,album_artist_mbids_text,album_artist_names_text,album_artist_types_text,film_soundtrack_composer_raw,composer_names_text,track_number,track_length_ms,recording_mbid,isrcs_text
1059,8290,"Hans Zimmer, Ryan Amon & Steve Mazzaro",The Black Sheep,Chappie,198184,Chappie,104399548,"Crime, Action, Science Fiction",tt1823672,2f4348cd-c7fe-436a-9fa3-1dee76743afb,...,False,e6de1f3b-6484-491c-88dd-6d619f142abc,Hans Zimmer,Person,Hans Zimmer,Hans Zimmer | Ryan Amon | Steve Mazzaro,7,268000.0,400c4f42-8487-4506-8c84-d65d2cef7e94,NaN
1060,8290,Hans Zimmer & Andrew Kawczynski,Welcome to the Real World,Chappie,198184,Chappie,104399548,"Crime, Action, Science Fiction",tt1823672,2f4348cd-c7fe-436a-9fa3-1dee76743afb,...,False,e6de1f3b-6484-491c-88dd-6d619f142abc,Hans Zimmer,Person,Hans Zimmer,Andrew Kawczynski | Hans Zimmer,6,232000.0,666a3c85-0931-4451-924a-3c25ff599e95,NaN
1061,8290,Hans Zimmer & Steve Mazzaro,It's a Dangerous City,Chappie,198184,Chappie,104399548,"Crime, Action, Science Fiction",tt1823672,2f4348cd-c7fe-436a-9fa3-1dee76743afb,...,False,e6de1f3b-6484-491c-88dd-6d619f142abc,Hans Zimmer,Person,Hans Zimmer,Hans Zimmer | Steve Mazzaro,1,129000.0,ad6159b6-e5e4-48ca-9356-86653939e10f,NaN
1062,8290,Hans Zimmer & Steve Mazzaro,You Lied to Me,Chappie,198184,Chappie,104399548,"Crime, Action, Science Fiction",tt1823672,2f4348cd-c7fe-436a-9fa3-1dee76743afb,...,False,e6de1f3b-6484-491c-88dd-6d619f142abc,Hans Zimmer,Person,Hans Zimmer,Hans Zimmer | Steve Mazzaro,11,246000.0,20bea297-6196-4919-8e90-06b00d0b3600,NaN
1063,8290,"Hans Zimmer, Steve Mazzaro & Andrew Kawczynski",Use Your Mind,Chappie,198184,Chappie,104399548,"Crime, Action, Science Fiction",tt1823672,2f4348cd-c7fe-436a-9fa3-1dee76743afb,...,False,e6de1f3b-6484-491c-88dd-6d619f142abc,Hans Zimmer,Person,Hans Zimmer,Andrew Kawczynski | Hans Zimmer | Steve Mazzaro,3,244000.0,5063ec8f-0ca9-4366-abeb-954a6ccaf123,NaN


In [11]:
# Total tracks where artist contains Danny Elfman

elfman_df = df_view[df_view["recording_artist_credit"].str.contains("Danny Elfman", case=False, na=False)]
print(f"Number of tracks by Danny Elfman: {len(elfman_df)}")
elfman_df.head()

Number of tracks by Danny Elfman: 385


,film_vote_count,recording_artist_credit,track_title,album_title,tmdb_id,film_title,film_revenue,film_genres,film_imdb_id,release_group_mbid,...,is_canonical_songtrack,album_artist_mbids_text,album_artist_names_text,album_artist_types_text,film_soundtrack_composer_raw,composer_names_text,track_number,track_length_ms,recording_mbid,isrcs_text
378,12222,Danny Elfman,Did That Hurt?,Fifty Shades of Grey: Original Motion Picture ...,216015,Fifty Shades of Grey,569651467,"Drama, Romance, Thriller",tt2322441,85852629-ce4c-4bb8-8d4a-50299d56a06c,...,True,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,Other,"Danny Elfman, Ellie Goulding, The Weeknd",Danny Elfman,16,174746.0,c2f7d647-f3af-41b2-8f74-2698a4eb4a8d,USQ4E1501387
381,12222,Danny Elfman,Ana and Christian,Fifty Shades of Grey: Original Motion Picture ...,216015,Fifty Shades of Grey,569651467,"Drama, Romance, Thriller",tt2322441,85852629-ce4c-4bb8-8d4a-50299d56a06c,...,True,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,Other,"Danny Elfman, Ellie Goulding, The Weeknd",Danny Elfman,15,204440.0,864c8ac9-9ab9-43b2-9f4e-439dc5ab8cc8,USQ4E1501383
804,12222,Danny Elfman,Clean You Up,Fifty Shades of Grey: Original Motion Picture ...,216015,Fifty Shades of Grey,569651467,"Drama, Romance, Thriller",tt2322441,fa7aa3a0-3de5-4273-bbb4-cfc13dbf9f1e,...,False,5b24fbab-c58f-4c37-a59d-ab232e2d98c4,Danny Elfman,Person,"Danny Elfman, Ellie Goulding, The Weeknd",Danny Elfman,9,163106.0,12a19c5f-806d-434a-b970-9fa4685d2cfb,USQ4E1501384
805,12222,Danny Elfman,Shades of Grey,Fifty Shades of Grey: Original Motion Picture ...,216015,Fifty Shades of Grey,569651467,"Drama, Romance, Thriller",tt2322441,fa7aa3a0-3de5-4273-bbb4-cfc13dbf9f1e,...,False,5b24fbab-c58f-4c37-a59d-ab232e2d98c4,Danny Elfman,Person,"Danny Elfman, Ellie Goulding, The Weeknd",Danny Elfman,1,127080.0,fa8db81e-bf01-4916-b01c-12acb6b5178b,USQ4E1501376
806,12222,Danny Elfman,Variations on a Shade,Fifty Shades of Grey: Original Motion Picture ...,216015,Fifty Shades of Grey,569651467,"Drama, Romance, Thriller",tt2322441,fa7aa3a0-3de5-4273-bbb4-cfc13dbf9f1e,...,False,5b24fbab-c58f-4c37-a59d-ab232e2d98c4,Danny Elfman,Person,"Danny Elfman, Ellie Goulding, The Weeknd",Danny Elfman,16,382906.0,f96b1122-0adc-436a-8038-a78a25ed9cd1,USQ4E1501391


In [12]:
# Unique tracks where artist contains Danny Elfman

elfman_df = df_view[
    df_view["recording_artist_credit"].str.contains("Danny Elfman", case=False, na=False)
].drop_duplicates(
    subset=["recording_artist_credit", "track_title"]
)

unique_elfman = len(elfman_df)
print(f"Number of unique Danny Elfman tracks: {unique_elfman}")
elfman_df.head()

Number of unique Danny Elfman tracks: 378


,film_vote_count,recording_artist_credit,track_title,album_title,tmdb_id,film_title,film_revenue,film_genres,film_imdb_id,release_group_mbid,...,is_canonical_songtrack,album_artist_mbids_text,album_artist_names_text,album_artist_types_text,film_soundtrack_composer_raw,composer_names_text,track_number,track_length_ms,recording_mbid,isrcs_text
378,12222,Danny Elfman,Did That Hurt?,Fifty Shades of Grey: Original Motion Picture ...,216015,Fifty Shades of Grey,569651467,"Drama, Romance, Thriller",tt2322441,85852629-ce4c-4bb8-8d4a-50299d56a06c,...,True,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,Other,"Danny Elfman, Ellie Goulding, The Weeknd",Danny Elfman,16,174746.0,c2f7d647-f3af-41b2-8f74-2698a4eb4a8d,USQ4E1501387
381,12222,Danny Elfman,Ana and Christian,Fifty Shades of Grey: Original Motion Picture ...,216015,Fifty Shades of Grey,569651467,"Drama, Romance, Thriller",tt2322441,85852629-ce4c-4bb8-8d4a-50299d56a06c,...,True,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,Other,"Danny Elfman, Ellie Goulding, The Weeknd",Danny Elfman,15,204440.0,864c8ac9-9ab9-43b2-9f4e-439dc5ab8cc8,USQ4E1501383
804,12222,Danny Elfman,Clean You Up,Fifty Shades of Grey: Original Motion Picture ...,216015,Fifty Shades of Grey,569651467,"Drama, Romance, Thriller",tt2322441,fa7aa3a0-3de5-4273-bbb4-cfc13dbf9f1e,...,False,5b24fbab-c58f-4c37-a59d-ab232e2d98c4,Danny Elfman,Person,"Danny Elfman, Ellie Goulding, The Weeknd",Danny Elfman,9,163106.0,12a19c5f-806d-434a-b970-9fa4685d2cfb,USQ4E1501384
805,12222,Danny Elfman,Shades of Grey,Fifty Shades of Grey: Original Motion Picture ...,216015,Fifty Shades of Grey,569651467,"Drama, Romance, Thriller",tt2322441,fa7aa3a0-3de5-4273-bbb4-cfc13dbf9f1e,...,False,5b24fbab-c58f-4c37-a59d-ab232e2d98c4,Danny Elfman,Person,"Danny Elfman, Ellie Goulding, The Weeknd",Danny Elfman,1,127080.0,fa8db81e-bf01-4916-b01c-12acb6b5178b,USQ4E1501376
806,12222,Danny Elfman,Variations on a Shade,Fifty Shades of Grey: Original Motion Picture ...,216015,Fifty Shades of Grey,569651467,"Drama, Romance, Thriller",tt2322441,fa7aa3a0-3de5-4273-bbb4-cfc13dbf9f1e,...,False,5b24fbab-c58f-4c37-a59d-ab232e2d98c4,Danny Elfman,Person,"Danny Elfman, Ellie Goulding, The Weeknd",Danny Elfman,16,382906.0,f96b1122-0adc-436a-8038-a78a25ed9cd1,USQ4E1501391


In [13]:
# Filter the dataframe by films with vote count >= 500

filtered_df = df_view[df_view["film_vote_count"] >= 500]

In [14]:
filtered_df['film_title'].nunique()

1613

In [15]:
# Determine the number of unique track-artist pairs in filtered dataframe

grouped_df = filtered_df.groupby([
    'track_title', 
    'recording_artist_credit'
    ]).count()

grouped_df.shape[0]

36148

In [16]:
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36580 entries, 66 to 78934
Data columns (total 25 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   film_vote_count               36580 non-null  int64  
 1   recording_artist_credit       36580 non-null  object 
 2   track_title                   36580 non-null  object 
 3   album_title                   36580 non-null  object 
 4   tmdb_id                       36580 non-null  int64  
 5   film_title                    36580 non-null  object 
 6   film_revenue                  36580 non-null  int64  
 7   film_genres                   36580 non-null  object 
 8   film_imdb_id                  36580 non-null  object 
 9   release_group_mbid            36580 non-null  object 
 10  release_mbid                  36580 non-null  object 
 11  rg_primary_type               36535 non-null  object 
 12  rg_secondary_types            36580 non-null  object 
 13  album

In [17]:
# Determine the number of unique tracks in filtered data for the major composers

filtered_williams_df = filtered_df[
    filtered_df["recording_artist_credit"].str.contains("John Williams", case=False, na=False)
].drop_duplicates(
    subset=["recording_artist_credit", "track_title"]
)
filtered_unique_williams = len(filtered_williams_df)


filtered_zimmer_df = filtered_df[
    filtered_df["recording_artist_credit"].str.contains("Hans Zimmer", case=False, na=False)
].drop_duplicates(
    subset=["recording_artist_credit", "track_title"]
)

filtered_unique_zimmer = len(filtered_zimmer_df)


filtered_elfman_df = filtered_df[
    filtered_df["recording_artist_credit"].str.contains("Danny Elfman", case=False, na=False)
].drop_duplicates(
    subset=["recording_artist_credit", "track_title"]
)

filtered_unique_elfman = len(filtered_elfman_df)

print(f"Number of unique John Williams tracks after filter: {filtered_unique_williams}")
print(f"Number of unique Hans Zimmer tracks after filter: {filtered_unique_zimmer}")
print(f"Number of unique Danny Elfman tracks after filter: {filtered_unique_elfman}")

Number of unique John Williams tracks after filter: 103
Number of unique Hans Zimmer tracks after filter: 489
Number of unique Danny Elfman tracks after filter: 378


In [18]:
# Calculate percentages of the remaining tracks after filtering

pct_williams = filtered_unique_williams / unique_williams
pct_zimmer = filtered_unique_zimmer / unique_zimmer
pct_elfman = filtered_unique_elfman / unique_elfman

print(f"Percentage of Williams tracks remaining after filtering {pct_williams:.0%}")
print(f"Percentage of Zimmer tracks remaining after filtering {pct_zimmer:.0%}")
print(f"Percentage of Elfman tracks remaining after filtering {pct_elfman:.0%}")

Percentage of Williams tracks remaining after filtering 100%
Percentage of Zimmer tracks remaining after filtering 85%
Percentage of Elfman tracks remaining after filtering 100%


In [19]:
filtered_df.sample(10)

,film_vote_count,recording_artist_credit,track_title,album_title,tmdb_id,film_title,film_revenue,film_genres,film_imdb_id,release_group_mbid,...,is_canonical_songtrack,album_artist_mbids_text,album_artist_names_text,album_artist_types_text,film_soundtrack_composer_raw,composer_names_text,track_number,track_length_ms,recording_mbid,isrcs_text
14503,4317,"Josef Schmidhuber, Chor des Bayerischen Rundfu...","Stabat Mater D383: I. ""Jesus Christus schwebt ...",The Killing of a Sacred Deer: Original Motion ...,399057,The Killing of a Sacred Deer,10700000,"Drama, Thriller, Mystery",tt5715874,48720389-7b38-4735-a4ce-6aee9324b4d5,...,False,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,Other,Unknown,Franz Schubert,1,181000.0,2c5968bc-ed71-4056-8849-d2d9fc660d10,NaN
24614,7868,Michael Abels,Ballet Memory,Us: Original Motion Picture Soundtrack,458723,Us,256071218,"Horror, Mystery",tt6857112,b7aa3ba9-4641-4dba-80ab-8400d536a043,...,False,6062f8e2-30ba-46ba-9cb7-07627697db28,Michael Abels,Person,Michael Abels,Michael Abels,5,71000.0,fa95c6be-aaa5-42d6-a486-97dab05dbb86,NaN
20683,3280,Mark Isham,Mom's Advice,The Longest Ride (Original Motion Picture Scor...,228205,The Longest Ride,63013281,"Drama, Romance",tt2726560,ceb69293-4458-4202-bff5-b72fae1a8808,...,False,56d6a50f-4be2-4796-9709-7eb88c45b63b,Mark Isham,Person,Mark Isham,Mark Isham,24,86000.0,f7003d49-04b0-4706-bdba-01e56ac00e70,NaN
68191,2567,Lorne Balfe,Police Swarm In,Carry‐On: Soundtrack from the Netflix Film,1005331,Carry-On,0,"Thriller, Action",tt21382296,2ba964b2-e565-41eb-8cfb-5e39215d7d74,...,False,0529a9e0-befb-4d9f-b6f9-1ccd7c83053e,Lorne Balfe,Person,Lorne Balfe,NaN,7,231930.0,0ef95675-48d8-48e3-90b8-a329c9f1b16a,NaN
18730,2661,Hauschka,Salvation,Adrift: Original Motion Picture Soundtrack,429300,Adrift,59945012,"Thriller, Romance, Adventure",tt6306064,214c3642-172f-4384-ab7d-7da9534b29e3,...,False,767026a6-9e39-463b-9d04-ed0f86ac5ee7,Hauschka,Person,Volker Bertelmann,NaN,17,325477.0,e1dfe200-038a-4cd9-a4e1-fe6a8999caa1,NaN
15888,4802,John Williams,Setting the Type,The Post,446354,The Post,179769467,"Drama, History",tt6294822,db02be29-ce96-4e5c-8960-bd6aa868ba80,...,False,53b106e7-0cc6-42cc-ac95-ed8d30a3a98e,John Williams,Person,John Williams,NaN,5,154306.0,31236b40-8e07-465f-aeab-0843c7d1aacf,USSM11710393
75788,1250,Carlo Siliotto,I'm In,Miracles From Heaven,339984,Miracles from Heaven,74000000,"Family, Drama",tt4257926,b21d2bbb-caa3-4402-a726-7215f51cbe98,...,False,7dd73d49-337a-490b-9372-da58246879ea,Carlo Siliotto,Person,Carlo Siliotto,Carlo Siliotto,4,NaN,aa104b73-9ad5-4c27-bf53-6d803f7a7059,NaN
41446,9334,John Murphy,Approaching the Beach,The Suicide Squad: Score from the Original Mot...,436969,The Suicide Squad,168717425,"Action, Comedy, Adventure",tt6334354,43053f73-8a92-409e-99fb-4a453d3e68de,...,False,e8aab2f0-164e-4984-a893-9e446f150202,John Murphy,Person,John Murphy,John Murphy,2,72187.0,5c72716b-2b99-4614-aa46-59c94498d0fb,USNLR2100924
31124,4520,Alfonso G. Aguilar,Look What You've Done,Klaus,508965,Klaus,0,"Animation, Family, Adventure, Comedy, Fantasy",tt4729430,bf14820f-0528-42d9-862a-ed840775a4f7,...,False,596e0e1a-2dbf-4384-be70-c4d368a81d88,Alfonso G. Aguilar,Person,Alfonso G. Aguilar,NaN,25,140000.0,64d6a1cc-45a2-4420-9893-2ca245b1c7c0,NaN
37293,11122,Trent Reznor and Atticus Ross,Portal,Soul: Original Motion Picture Score,508442,Soul,121977511,"Animation, Family, Comedy, Fantasy, Drama, Music",tt2948372,86156f0f-9ce8-439b-a76b-e14d63f5366a,...,False,bc880664-6751-4998-984d-ebdbff15d229 | 1fefa8b...,Trent Reznor | Atticus Ross,Person | Person,"Jon Batiste, Trent Reznor, Atticus Ross",Atticus Ross | Trent Reznor,7,NaN,cd045a28-fae4-4ca1-8de3-3d10a31e2b89,USWD12098509


In [20]:
# Filtered dataframe sorted by revenue

filtered_df = (
    filtered_df
        .sort_values(by="film_revenue", ascending=False)
        .drop_duplicates(subset="film_title")
)
filtered_df

,film_vote_count,recording_artist_credit,track_title,album_title,tmdb_id,film_title,film_revenue,film_genres,film_imdb_id,release_group_mbid,...,is_canonical_songtrack,album_artist_mbids_text,album_artist_names_text,album_artist_types_text,film_soundtrack_composer_raw,composer_names_text,track_number,track_length_ms,recording_mbid,isrcs_text
25312,27141,Alan Silvestri,Tres Amigos,Avengers: Endgame (Original Motion Picture Sou...,299534,Avengers: Endgame,2799439100,"Adventure, Science Fiction, Action",tt4154796,2261260c-2900-4795-8f18-99343ac13714,...,False,f4c53778-e48d-4da4-9ddd-cc007f715d64,Alan Silvestri,Person,Alan Silvestri,Alan Silvestri,25,218000.0,e863ef9a-fc61-44dc-b0ea-b1d2a677700a,USHR11939148
53197,13547,Simon Franglen,Songcord Opening,Avatar: The Way of Water (Original Score),76600,Avatar: The Way of Water,2353096253,"Action, Adventure, Science Fiction",tt1630029,f9344821-dc34-482b-b1c9-b3bc71ce9f26,...,False,c82beb12-579c-48e6-aa97-dd5b0f3a6078,Simon Franglen,Person,Simon Franglen,NaN,2,118385.0,e1846395-9374-4493-b3dc-01a40e75b80c,USHR12244994
2561,20206,John Williams,The Starkiller,Star Wars: The Force Awakens: Original Motion ...,140607,Star Wars: The Force Awakens,2068223624,"Adventure, Action, Science Fiction",tt2488496,405fd3c5-0a45-456a-b853-6f734d3b57aa,...,True,53b106e7-0cc6-42cc-ac95-ed8d30a3a98e,John Williams,Person,John Williams,John Williams,12,111506.0,b6544026-2b26-4251-8f3a-256dde56e9c0,USWD11575432
18177,31351,Alan Silvestri,What More Could I Lose?,Avengers: Infinity War,299536,Avengers: Infinity War,2052415039,"Adventure, Action, Science Fiction",tt4154756,1197ddb7-5a62-49c5-ad8e-10ae21b0212a,...,False,f4c53778-e48d-4da4-9ddd-cc007f715d64,Alan Silvestri,Person,Alan Silvestri,NaN,10,216000.0,a4a861d8-f10e-49e5-9557-51d121b2c191,USHR11838556
77807,1113,Michael Giacchino,Snake Away Pt. 1,Zootopia 2 (Original Motion Picture Soundtrack),1084242,Zootopia 2,1703000000,"Animation, Comedy, Adventure, Family, Mystery",tt26443597,60748f4f-b6ae-49c1-8374-46d9352c14d3,...,False,150e14bd-915d-4574-b574-f77dd7ea72ad,Michael Giacchino,Person,Michael Giacchino,NaN,7,135253.0,32615e76-91ba-4802-a18c-a47c962595f4,USWD12538810
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35766,3564,Pokey LaFarge & Harry Melling,Washed in the Blood,The Devil All the Time,499932,The Devil All the Time,0,"Crime, Drama, Thriller",tt7395114,453c9a29-b225-4a9f-a885-4ea91d3657c9,...,False,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,Other,"Danny Bensi, Saunder Jurriaans",NaN,2,199000.0,4d4805a9-2ab8-4f06-b2d7-707ffd02384e,NaN
60020,664,Jess Ribeiro,Strange Game,Love At First Sight,353577,Love at First Sight,0,"Romance, Drama",tt2183014,b5dc4421-707f-4f4e-92a8-f2026904ff85,...,False,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,Other,Paul Saunderson,NaN,11,139635.0,fb31c897-300c-4f31-bb9f-ab85b8c7ba4b,NaN
35812,564,Joseph Trapanese,Flipped Car,Spontaneous,604578,Spontaneous,0,"Comedy, Horror",tt5774062,851cbf6a-c83a-4d70-8053-31b8bbb9a4cb,...,False,70dbaf82-6cad-47b8-ae79-b36ef83d5ee3,Joseph Trapanese,Person,Unknown,Joseph Trapanese,10,87000.0,44d5350b-d011-43bd-a440-464dc1c04bef,NaN
35656,756,Brooke Blair & Will Blair,Kitchen Combat,The Silencing (Original Motion Picture Soundtr...,603119,The Silencing,0,"Thriller, Action, Crime",tt7149730,b37bf4de-29c4-4584-8cea-f98aaa1be94c,...,False,7d0efe0b-a1a5-47fb-a7f3-fae944561f23 | b7a4c86...,Brooke Blair | Will Blair,Person | Person,Unknown,NaN,7,138000.0,1266e892-8779-4761-8222-9d7c918f3c5b,NaN


In [21]:
excluded_films_df = df_view[df_view["film_vote_count"] < 500]

In [22]:
excluded_films_df['film_title'].nunique()

2782

In [23]:
excluded_films_df.sample(10)

,film_vote_count,recording_artist_credit,track_title,album_title,tmdb_id,film_title,film_revenue,film_genres,film_imdb_id,release_group_mbid,...,is_canonical_songtrack,album_artist_mbids_text,album_artist_names_text,album_artist_types_text,film_soundtrack_composer_raw,composer_names_text,track_number,track_length_ms,recording_mbid,isrcs_text
9586,53,Bibi & Tina,Una,Tohuwabohu total (Der Original-Soundtrack zum ...,410791,Bibi & Tina: Perfect Pandemonium,0,"Comedy, Fantasy, Adventure, Family",tt5950646,de21c530-191d-4f05-b9dd-86c9f63790cf,...,False,3207a596-387d-47af-9d37-0200735c64a5,Bibi & Tina,Other,Unknown,NaN,11,156000.0,28d4d01c-0b63-45b3-a956-47352fd799b2,DEB581600464
51124,41,Vnature,Triceratops Machete 2,Stop‐Zemlia (Original Motion Picture Soundtrack),794785,Stop-Zemlia,0,Drama,tt14028890,08d5b252-432f-4712-8518-bf3122de7b64,...,False,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,Other,Unknown,NaN,20,266000.0,6bb0a134-46c3-4b31-9413-c404478fd46c,TCAFZ2242592
40066,93,Nathan Halpern,Stuffed With Toys,The Lovers and the Despot (Original Motion Pic...,426253,The Lovers,0,"Comedy, Drama, Romance",tt5770620,b6db5a4a-9d46-4bbb-b99f-d6f82651ff2a,...,False,e9ffed3e-2976-420f-a2e0-462aa20c6458,Nathan Halpern,Person,Mandy Hoffman,NaN,9,196760.0,96082596-9dd9-42be-8111-915bb631c5f7,NaN
39981,272,StarKid Productions,Black Friday,Black Friday,765869,Black Friday,0,"Horror, Comedy, Science Fiction",tt11649338,0ba72d47-8c1d-41c7-8364-2099ebbde92f,...,False,59da61e2-c75e-428c-9f4a-02e6cda0fe9a,StarKid Productions,Group,Patrick Stump,NaN,14,218947.0,bfa548a7-72d3-4374-a1d0-f420a791a914,TCAER2043732
58673,10,Rashmeet Kaur & Shashwat Sachdev,Dil Hai Ranjhana,Tejas,737370,Tejas,0,Drama,tt6950476,3818cfe3-2dc2-4459-b40e-80281b22be8b,...,False,26da2fd6-329a-4b0a-b5c2-35e73eaa36fc,Shashwat Sachdev,Person,Unknown,Shashwat Sachdev,1,211000.0,58ba209e-a569-4992-8b8f-ff88bf19ddf0,INZ031413748
48973,394,Bertrand Blessing,Gestation,Goliath,884971,Goliath,0,"Thriller, Drama",tt13399946,4361c357-593a-4ffd-a9ce-a658933557d2,...,False,d19aefa6-b93c-40b0-b9b8-b9eaf349be85,Bertrand Blessing,Person,"Christophe La Pinta, Bertrand Blessing",NaN,14,162000.0,f3587eb4-8e90-457e-a073-435764368fde,NaN
75378,84,Jim Williams,Blood with Blood,Alpha: Original Motion Picture Soundtrack,1284460,Alpha,0,"Horror, Drama, Science Fiction",tt32275943,32970442-4dd9-4762-a688-f7fa9f7b36f8,...,False,7f0a510c-a5d9-47e7-badc-5ca60c46ac12,Jim Williams,Person,Jim Williams,NaN,9,235693.0,e7bb8d05-437d-4f2b-b3f7-d801821a9902,FR9W12543742
28061,52,David James Nielsen,The Ontario Ship,Annabelle Hooper and the Ghosts of Nantucket: ...,410284,Annabelle Hooper and the Ghosts of Nantucket,0,"TV Movie, Mystery, Family",tt3032060,072ae9b0-fcfc-416f-b4ba-4ccdc2b0c182,...,False,f2cbd1cd-5357-4212-b612-05e81e3ae69e,David James Nielsen,NaN,David James Nielsen,NaN,11,366000.0,cca6632e-cb56-4d3c-a148-efa78ba94d56,NaN
63872,396,Amelia Warner,The Channel Plan,Young Woman and the Sea: Original Score,774531,Young Woman and the Sea,581725,"History, Drama",tt5177114,43426819-43da-4813-88f6-72f33edbc6c6,...,False,1048fc96-34c1-43c6-877b-08606f7017a1,Amelia Warner,Person,Amelia Warner,NaN,9,155000.0,d771d77b-a263-461c-93dd-f37c872d85ad,USWD12429947
42347,181,Laura Rossi,Gabriel’s Guilt,Hurricane: Original Motion Picture Soundtrack,506863,Hurricane,2137751,"War, Drama, Action",tt7515456,a875daec-e68f-40a1-870d-417604adadd4,...,False,4b04af6b-9a1b-4f1d-bbe1-d79d7a5f678e,Laura Rossi,Person,Laura Rossi,NaN,15,56286.0,8ff235c5-806e-4ac6-823b-943a149c86b7,NaN


In [24]:
# Excluded films sorted by revenue

excluded_films_df = (
    excluded_films_df
        .sort_values(by="film_revenue", ascending=False)
        .drop_duplicates(subset="film_title")
)

excluded_films_df

,film_vote_count,recording_artist_credit,track_title,album_title,tmdb_id,film_title,film_revenue,film_genres,film_imdb_id,release_group_mbid,...,is_canonical_songtrack,album_artist_mbids_text,album_artist_names_text,album_artist_types_text,film_soundtrack_composer_raw,composer_names_text,track_number,track_length_ms,recording_mbid,isrcs_text
69017,309,"Andrew Kawczynski, Rupert Gregson-Williams",True Gold Fears No Fire,The Eight Hundred,508935,The Eight Hundred,461421559,"War, History, Drama, Action",tt7294150,ab08cf3d-6d6f-47ad-9bb7-35769f557652,...,False,9a4c8169-5d2d-4f34-9942-cbcc49564c61 | 388f971...,Andrew Kawczynski | Rupert Gregson-Williams,Person | Person,"Rupert Gregson-Williams, Andrew Kawczynski",Andrew Kawczynski,1,279000.0,9afa3ff1-7c3f-4f8b-a7dd-20c5aa603322,NaN
78743,427,Theodore Shapiro & Caroline Shaw,Acute Psychosis,"The Housemaid, Vol. 1: Original Motion Picture...",1368166,The Housemaid,245700000,"Mystery, Thriller",tt27543632,bf5aec28-3597-4fae-847f-03d372690b5c,...,True,e91c7817-73af-4bd8-a707-71e41aa6aa27,Theodore Shapiro,Person,Theodore Shapiro,NaN,7,159760.0,c791ee8b-5227-467c-b256-904286d4ade0,USLS52596407
69289,121,Sam C.S.,Face Of Humiliation,Pushpa 2 The Rule Ost,857598,Pushpa 2 - The Rule,219000000,"Action, Drama, Thriller, Crime",tt16539454,957ca3af-b0f4-4a62-85dd-99bbc69ea741,...,False,f7645b5f-6462-4057-bfd6-d196e2a2a30d,Sam C.S.,Person,"Devi Sri Prasad, Sam C S",NaN,11,36139.0,2c66da86-03da-44e8-b13e-9f9619ee306e,NaN
6137,107,Christopher Young,"Baigujing, Lady White Bone",The Monkey King 2,381902,The Monkey King 2,193677158,"Action, Adventure, Fantasy",tt4591310,4235e895-fcbb-4115-8bfb-10162e2576a6,...,False,53d8a593-c5e9-4af8-9657-d3bf19b1bb89,Christopher Young,Person,Christopher Young,Christopher Young,4,332000.0,17c05da5-243d-4f30-9d76-8571c587e40f,NaN
61254,26,Ramya Behara & Rahul Nambiar,Rang De,A Aa,372399,A Aa,189432924,"Family, Comedy, Romance, Drama",tt5684466,38ae81ab-ef70-46c2-b516-4c2174943b28,...,False,0964ac05-6458-493a-a6a4-bc4164b1641f,Mickey J. Meyer,Person,Unknown,NaN,2,241000.0,8dc0abc6-6452-4e03-bba9-1df125a8f311,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44743,20,Benji Merrison,Feeling And Heart,The Beatles And India,820453,The Beatles and India,0,"Documentary, Music",tt14736144,47a6a6bd-5c3d-4bce-821f-6f30d089804c,...,False,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,Other,Unknown,NaN,13,112000.0,e970e26f-a92b-4e55-a1b5-8a494919ee3d,NaN
17621,13,Santhosh Narayanan,Address Song,Meyaadha Maan,463293,Meyaadha Maan,0,"Romance, Comedy",tt7531040,1d2e6429-f57c-45f0-b148-b82255c900d0,...,False,e2e0ce51-9b3a-485f-bd51-d58a5b624341 | 5503136...,Pradeep Kumar | Santhosh Narayanan,Person | Person,"Santhosh Narayanan, Pradeep Kumar",NaN,2,271281.0,6eeacb33-a277-4b58-b3f0-c6567d6bbfdc,NaN
17622,23,Alexandre Azaria,Like a Ghost in the Winter,Par instinct,442267,Par instinct,0,"Drama, TV Movie",tt5628688,44a87e40-b354-4e43-9045-53b0c50411dc,...,False,d2624184-416c-4ace-8333-efe15fce93a0,Alexandre Azaria,Person,Unknown,NaN,13,144000.0,e556d4c2-65e7-4cf8-9221-6728d37df21f,NaN
44595,23,AJ Hochhalter,Whiskey Boomed,Neat: The Story of Bourbon,506730,Neat: The Story of Bourbon,0,Documentary,tt7109844,5f842aea-2eff-4536-8646-5d0af351d0fd,...,False,ab3a2927-5230-44f3-a79a-c33aff0c5ae1,AJ Hochhalter,Person,Unknown,NaN,4,181000.0,bf00bb4e-21f6-4dde-a14c-4ac609f5f448,NaN


In [25]:
# Adding flag for vote count >= 500 to the original dataframe (all columns)

df['vote_count_above_500'] = df['film_vote_count'] >= 500
df.head()

,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,film_is_horror,film_is_music,film_is_mystery,film_is_romance,film_is_science_fiction,film_is_tv_movie,film_is_thriller,film_is_war,film_is_western,vote_count_above_500
0,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,True,False,False,False,False,False,True,False,False,False
1,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,True,False,False,False,False,False,True,False,False,False
2,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,False,False,False,False,False,False,False,False
3,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,False,False,False,False,False,False,False,False
4,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,False,False,False,False,False,False,False,False,False,False


In [26]:
len(df[df['vote_count_above_500'] == True])

36580

In [27]:
# Exporting wide dataframe with all columns to CSV

out_path = "./pipeline/3.6.Wide_vote_count_analysis.csv"

df.to_csv(out_path, index=False)

In [28]:
# Adding vote count flag to the album CSV

filepath = '/work/pipeline/3.5.Albums_exploded_genre.csv'
df = pd.read_csv(filepath)
df['vote_count_above_500'] = df['film_vote_count'] >= 500

out_path = "./pipeline/3.6.Albums_vote_count_analysis.csv"
df.to_csv(out_path, index=False)

In [29]:
# Artist and track CSV's carry over

import shutil

shutil.copy(
    "./pipeline/3.5.Artists_exploded_genre.csv",
    "./pipeline/3.6.Artists_vote_count_analysis.csv"
)

shutil.copy(
    "./pipeline/3.5.Tracks_exploded_genre.csv",
    "./pipeline/3.6.Tracks_vote_count_analysis.csv"
)

'./pipeline/3.6.Tracks_vote_count_analysis.csv'

# 

### 

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b124131d-2024-4253-bb46-8043aed4b78f' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>